# Tarea 2 – Parte 1: Corpus y extracción (OCR + ASR)

**Pregunta guía:** ¿Pueden convertirse documentos escaneados y entrevistas en audio en un archivo de texto estructurado y buscable?

## 1.1. Configuración inicial

Registrar en la primera celda Markdown la fecha de extracción, las fuentes y licencias, y las versiones de PaddleOCR y del modelo ASR (usen los entornos de las sesiones: uv sync --group docai y uv sync --group audio).

- **Fecha de extracción:** 22 de octubre de 2026
- **Fuentes y licencias:** (por completar)
- **Versiones:** Python 3.12.14 · PaddleOCR 3.7.0 (PaddlePaddle 3.3.1, modelos PP-OCRv6 medium, `lang="es"`) · faster-whisper 1.2.1 (modelo `small`, int8, CPU) · pyannote-audio 4.0.7 · torch 2.14.1

In [14]:
import os
import json
import time
from importlib.metadata import version
from pathlib import Path

# Se evita que PaddleOCR revise la conexión con cada servidor de modelos (va antes de importarlo)
os.environ["PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK"] = "True"

import torch  # Se importa antes que paddleocr: en Windows, al revés, torch falla (WinError 127)
import pandas as pd
import numpy as np
import soundfile as sf
from paddleocr import PaddleOCR
from faster_whisper import WhisperModel, decode_audio

for paquete in ["paddleocr", "paddlepaddle", "faster-whisper", "pyannote-audio", "torch"]:
    print(f"{paquete}: {version(paquete)}")


paddleocr: 3.7.0
paddlepaddle: 3.3.1
faster-whisper: 1.2.1
pyannote-audio: 4.0.7
torch: 2.14.1


In [4]:
# Se crea el lector OCR en español (modelos PP-OCRv6).
# enable_mkldnn=False evita el error de oneDNN en procesadores Intel.
# El clasificador de orientación endereza las fotos giradas 90°, y el texto se busca en una copia
# de máximo 1600 px para que las fotos de 12 MP no tarden más de 10 minutos cada una en esta laptop.
ocr = PaddleOCR(
    lang="es",
    enable_mkldnn=False,
    use_doc_orientation_classify=True,    # se endereza la página si está girada
    use_doc_unwarping=False,              # no se "aplana" la hoja
    use_textline_orientation=False,       # no se gira cada línea
    text_det_limit_type="max",
    text_det_limit_side_len=1600,         # lado mayor de la copia donde se busca el texto
)

Creating model: ('PP-LCNet_x1_0_doc_ori', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\LENOVO\.paddlex\official_models\PP-LCNet_x1_0_doc_ori`.
Creating model: ('PP-OCRv6_medium_det', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\LENOVO\.paddlex\official_models\PP-OCRv6_medium_det`.
Creating model: ('PP-OCRv6_medium_rec', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\LENOVO\.paddlex\official_models\PP-OCRv6_medium_rec`.


In [3]:
# Se carga faster-whisper "small" en CPU con int8 (la misma configuración de la línea base de la Parte 2)
modelo_asr = WhisperModel("small", device="cpu", compute_type="int8")

## 1.2. OCR (Sesión 10):

Extraer texto de los 25 documentos con PaddleOCR (impresos, fotos y manuscritos),guardando respaldo crudo y recortes en datos/raw/documentos/. Documentar parámetros (idioma,preprocesamiento).

A continuación, se describen los documentos selecciones:

1. 8 imágenes imprenta: Corresponde al documento de mi titulación.
2. 8 fotos de un documento imprenta: Corresponde al docuemento de SERVIR "Estudio de gobiernos locales 2016: Principales hallazgos sobre los puestos con los que cuentan y las personas que los ocupan" (https://repositorio.servir.gob.pe/item/025583e3-60dc-4f56-91d3-9160138f24b8). En la página 4, indica que se puede usar este documento con fines de estudios privados, siempre que se reconozca la autoría de SERVIR.
3. 9 fotos de manuscritos: Corresponde a mis apuntes de clase de estadística, matemática y cerámica.

**Parámetros del OCR:** español (`lang="es"`, modelos PP-OCRv6 medium); clasificador de orientación activado para enderezar las fotos giradas 90°; el texto se busca en una copia de máximo 1600 px, porque en esta laptop (CPU de 2 núcleos) cada foto de 12 MP tardaba de 9 a 13 minutos y así tarda cerca de 3; oneDNN desactivado por un error de PaddlePaddle en procesadores Intel. No se aplica otro preprocesamiento. Los respaldos crudos (JSON con texto, cajas y confianza, e imagen con las cajas detectadas) se guardan en `datos/raw/documentos/ocr/`.

In [6]:
# Se listan los 25 documentos y se asigna su tipo según el número: 001–008 impresos, 009–016 fotos y 017–025 manuscritos
CARPETA_DOCS = Path("datos/raw/documentos")
CARPETA_OCR = CARPETA_DOCS / "ocr"          # respaldo crudo del OCR
CARPETA_OCR.mkdir(exist_ok=True)

def tipo_documento(numero):
    if numero <= 8:
        return "impreso"
    elif numero <= 16:
        return "foto"
    return "manuscrito"

archivos = sorted(CARPETA_DOCS.glob("doc_*.*"))
documentos = pd.DataFrame({"id": [f.stem for f in archivos], "archivo": [f.as_posix() for f in archivos]})
documentos["tipo"] = documentos["id"].str[-3:].astype(int).apply(tipo_documento)
documentos["tipo"].value_counts()

tipo
manuscrito    9
impreso       8
foto          8
Name: count, dtype: int64

**Excepción:** en la primera ejecución, el clasificador de orientación giró por error 180° la foto `doc_009` (inclinada y con una franja gris oscura arriba): su imagen enderezada quedó de cabeza y el texto salió ilegible (confianza media 0,64). Por eso, ese documento se procesa sin el clasificador; el resto de la configuración es la misma.

In [8]:
# Segundo lector, igual al primero pero sin el clasificador de orientación, solo para doc_009
SIN_ORIENTACION = ["doc_009"]
ocr_sin_orientacion = PaddleOCR(
    lang="es",
    enable_mkldnn=False,
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_textline_orientation=False,
    text_det_limit_type="max",
    text_det_limit_side_len=1600,
)

Creating model: ('PP-OCRv6_medium_det', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\LENOVO\.paddlex\official_models\PP-OCRv6_medium_det`.
Creating model: ('PP-OCRv6_medium_rec', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\LENOVO\.paddlex\official_models\PP-OCRv6_medium_rec`.


In [12]:
# Se aplica el OCR a cada documento y se guarda su respaldo crudo (JSON e imágenes) en datos/raw/documentos/ocr/.
# Si un documento ya tiene su JSON, se lee de ahí y no se vuelve a procesar: así, al ejecutar de nuevo el notebook,
# no se repite la hora de OCR.
textos = []
for archivo in documentos["archivo"]:
    respaldo = CARPETA_OCR / f"{Path(archivo).stem}_res.json"
    if respaldo.exists():
        print(f"{Path(archivo).name}: se lee el respaldo")
    else:
        inicio = time.perf_counter()
        lector = ocr_sin_orientacion if Path(archivo).stem in SIN_ORIENTACION else ocr
        resultado = lector.predict(archivo)[0]
        resultado.save_to_json(CARPETA_OCR)
        resultado.save_to_img(CARPETA_OCR)
        print(f"{Path(archivo).name}: {time.perf_counter() - inicio:.0f} s")
    with open(respaldo, encoding="utf-8") as f:
        textos.append("\n".join(json.load(f)["rec_texts"]))

documentos["texto_ocr"] = textos
documentos["modelo_ocr"] = "PaddleOCR 3.7.0 (PP-OCRv6 medium, lang=es, orientación activada, detección máx. 1600 px)"
documentos.loc[documentos["id"].isin(SIN_ORIENTACION), "modelo_ocr"] = "PaddleOCR 3.7.0 (PP-OCRv6 medium, lang=es, orientación desactivada, detección máx. 1600 px)"

doc_001.png: se lee el respaldo
doc_002.png: se lee el respaldo
doc_003.png: se lee el respaldo
doc_004.png: se lee el respaldo
doc_005.png: se lee el respaldo
doc_006.png: se lee el respaldo
doc_007.png: se lee el respaldo
doc_008.png: se lee el respaldo
doc_009.jpg: 191 s
doc_010.jpg: se lee el respaldo
doc_011.jpg: se lee el respaldo
doc_012.jpg: se lee el respaldo
doc_013.jpg: se lee el respaldo
doc_014.jpg: se lee el respaldo
doc_015.jpg: se lee el respaldo
doc_016.jpg: se lee el respaldo
doc_017.jpg: se lee el respaldo
doc_018.jpg: se lee el respaldo
doc_019.jpg: se lee el respaldo
doc_020.jpg: se lee el respaldo
doc_021.jpg: se lee el respaldo
doc_022.jpg: se lee el respaldo
doc_023.jpg: se lee el respaldo
doc_024.jpg: se lee el respaldo
doc_025.jpg: se lee el respaldo


## 1.3. ASR (Sesión 13)

Transcribir los 30 minutos con voz.py o equivalente: VAD, transcripción con faster-whisper, diarización en al menos un audio multihablante y exportación a CSV, SRT y TXT. Guardar losarchivos originales y salidas pesadas en datos/raw/audio/
(sin versionarlos en Git).




**Preparación de los clips:** cada grabación original (en `datos/raw/audio/originales/`) se convierte a 16 kHz mono, el formato que usan los modelos de ASR, y se guarda como `clip_XXX.wav`. Las grabaciones cortas quedan como un solo clip, y la conversación de 10 minutos con un compañero de trabajo (2 hablantes, con su autorización verbal al inicio de la grabación) se divide en 8 partes, cortando en las pausas para no partir palabras.

In [15]:
# Se preparan los clips: cada original se convierte a 16 kHz mono y se guarda como clip_XXX.wav
CARPETA_AUDIO = Path("datos/raw/audio")
CARPETA_ORIGINALES = CARPETA_AUDIO / "originales"
SR = 16_000   # muestras por segundo

# Grabación original -> número de partes (el orden define el número de cada clip)
ORIGINALES = {
    "Grabación estándar 1.mp3": 1,
    "Grabación estándar 2.mp3": 1,
    "Grabación estándar 3.mp3": 1,
    "Grabación estándar 4.mp3": 1,
    "Grabación estándar 5.mp3": 1,
    "Grabación estándar 6.mp3": 1,
    "conversacion_trabajo.mp3": 8,
}

def punto_de_corte(audio, segundo, ventana=10):
    """Busca el tramo de 0,1 s más silencioso a ±10 s del segundo indicado y devuelve su posición."""
    tramo = int(0.1 * SR)
    inicio = max(0, int((segundo - ventana) * SR))
    fin = min(len(audio), int((segundo + ventana) * SR))
    energias = [np.abs(audio[i:i + tramo]).mean() for i in range(inicio, fin - tramo, tramo)]
    return inicio + int(np.argmin(energias)) * tramo

def cortar_en_partes(audio, n_partes):
    """Divide el audio en n partes de duración parecida, cortando en pausas."""
    duracion = len(audio) / SR
    cortes = [0] + [punto_de_corte(audio, duracion * k / n_partes) for k in range(1, n_partes)] + [len(audio)]
    return [audio[a:b] for a, b in zip(cortes[:-1], cortes[1:])]

filas = []
for original, n_partes in ORIGINALES.items():
    audio = decode_audio(str(CARPETA_ORIGINALES / original), sampling_rate=SR)
    for parte in cortar_en_partes(audio, n_partes):
        id_clip = f"clip_{len(filas) + 1:03d}"
        archivo = CARPETA_AUDIO / f"{id_clip}.wav"
        sf.write(archivo, parte, SR)
        filas.append({"id": id_clip, "archivo_audio": archivo.as_posix(), "original": original,
                      "duracion_seg": round(len(parte) / SR, 1)})

clips = pd.DataFrame(filas)
print(f"{len(clips)} clips, {clips['duracion_seg'].sum() / 60:.1f} minutos")
clips


14 clips, 19.0 minutos


,id,archivo_audio,original,duracion_seg
0,clip_001,datos/raw/audio/clip_001.wav,Grabación estándar 1.mp3,53.7
1,clip_002,datos/raw/audio/clip_002.wav,Grabación estándar 2.mp3,74.5
2,clip_003,datos/raw/audio/clip_003.wav,Grabación estándar 3.mp3,133.0
3,clip_004,datos/raw/audio/clip_004.wav,Grabación estándar 4.mp3,93.7
4,clip_005,datos/raw/audio/clip_005.wav,Grabación estándar 5.mp3,72.9
5,clip_006,datos/raw/audio/clip_006.wav,Grabación estándar 6.mp3,105.5
6,clip_007,datos/raw/audio/clip_007.wav,conversacion_trabajo.mp3,75.3
7,clip_008,datos/raw/audio/clip_008.wav,conversacion_trabajo.mp3,80.6
8,clip_009,datos/raw/audio/clip_009.wav,conversacion_trabajo.mp3,79.8
9,clip_010,datos/raw/audio/clip_010.wav,conversacion_trabajo.mp3,67.8


**Transcripción:** faster-whisper `small` (int8, CPU) con VAD activado (el detector de voz Silero que trae faster-whisper, para que Whisper no "alucine" frases en los silencios) e idioma fijado en español. Por cada clip se guarda en `datos/raw/audio/transcripciones/` un JSON con los segmentos (inicio, fin y texto) y el tiempo de proceso, más su versión en SRT (subtítulos) y TXT.

In [16]:
# Carpeta del respaldo crudo de las transcripciones (JSON, SRT y TXT por clip)
CARPETA_ASR = CARPETA_AUDIO / "transcripciones"
CARPETA_ASR.mkdir(exist_ok=True)

def tiempo_srt(segundos):
    """Convierte segundos al formato de tiempo de los subtítulos SRT (HH:MM:SS,mmm)."""
    ms = int(round(segundos * 1000))
    h, ms = divmod(ms, 3_600_000)
    m, ms = divmod(ms, 60_000)
    s, ms = divmod(ms, 1000)
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"

def a_srt(segmentos):
    """Arma el texto SRT: número, intervalo de tiempo y texto de cada segmento."""
    bloques = [f"{i}\n{tiempo_srt(s['inicio'])} --> {tiempo_srt(s['fin'])}\n{s['texto']}\n"
               for i, s in enumerate(segmentos, 1)]
    return "\n".join(bloques)

# Se transcribe cada clip. Si ya tiene su JSON, se salta: así, al agregar grabaciones nuevas, solo se procesan esas.
for fila in clips.itertuples():
    respaldo = CARPETA_ASR / f"{fila.id}.json"
    if respaldo.exists():
        print(f"{fila.id}: se lee el respaldo")
        continue
    inicio = time.perf_counter()
    segmentos, info = modelo_asr.transcribe(fila.archivo_audio, language="es", vad_filter=True)
    segmentos = [{"inicio": round(s.start, 2), "fin": round(s.end, 2), "texto": s.text.strip()} for s in segmentos]
    segundos = time.perf_counter() - inicio
    resultado = {"id": fila.id, "idioma": info.language, "duracion_seg": fila.duracion_seg,
                 "segundos_proceso": round(segundos, 1), "segmentos": segmentos}
    respaldo.write_text(json.dumps(resultado, ensure_ascii=False, indent=1), encoding="utf-8")
    (CARPETA_ASR / f"{fila.id}.srt").write_text(a_srt(segmentos), encoding="utf-8")
    (CARPETA_ASR / f"{fila.id}.txt").write_text(" ".join(s["texto"] for s in segmentos), encoding="utf-8")
    print(f"{fila.id}: {fila.duracion_seg:.0f} s de audio en {segundos:.0f} s (RTF {segundos / fila.duracion_seg:.2f})")

clip_001: 54 s de audio en 54 s (RTF 1.01)
clip_002: 74 s de audio en 86 s (RTF 1.15)
clip_003: 133 s de audio en 179 s (RTF 1.35)
clip_004: 94 s de audio en 176 s (RTF 1.88)
clip_005: 73 s de audio en 84 s (RTF 1.15)
clip_006: 106 s de audio en 115 s (RTF 1.09)
clip_007: 75 s de audio en 74 s (RTF 0.98)
clip_008: 81 s de audio en 86 s (RTF 1.07)
clip_009: 80 s de audio en 68 s (RTF 0.85)
clip_010: 68 s de audio en 58 s (RTF 0.85)
clip_011: 73 s de audio en 85 s (RTF 1.16)
clip_012: 86 s de audio en 84 s (RTF 0.98)
clip_013: 59 s de audio en 125 s (RTF 2.12)
clip_014: 85 s de audio en 64 s (RTF 0.75)


In [17]:
# Se leen los respaldos y se agregan a la tabla de clips el idioma, el texto y el tiempo de proceso
def leer_respaldo(id_clip):
    with open(CARPETA_ASR / f"{id_clip}.json", encoding="utf-8") as f:
        return json.load(f)

respaldos = [leer_respaldo(i) for i in clips["id"]]
clips["idioma"] = [r["idioma"] for r in respaldos]
clips["texto_asr"] = [" ".join(s["texto"] for s in r["segmentos"]) for r in respaldos]
clips["segundos_asr"] = [r["segundos_proceso"] for r in respaldos]
clips["modelo_asr"] = "faster-whisper 1.2.1 small (int8, CPU, VAD activado, idioma es)"
clips[["id", "duracion_seg", "segundos_asr", "texto_asr"]]

,id,duracion_seg,segundos_asr,texto_asr
0,clip_001,53.7,54.1,"Harry, cuéntame tu experiencia con esa vez que..."
1,clip_002,74.5,85.6,"Sobre tu experiencia en Uruguay, ¿qué es lo má..."
2,clip_003,133.0,179.1,"Y sobre tu experiencia viviendo en Uruguay, ¿q..."
3,clip_004,93.7,175.7,¿Y qué habilidades has aprendido viviendo allá...
4,clip_005,72.9,84.0,"¡Jarri, te acuerdas cuando mi tía Loti una vez..."
5,clip_006,105.5,115.3,"También la experiencia conmigo. Bueno, la frec..."
6,clip_007,75.3,74.1,"Como te comentaba, esto es para fines académic..."
7,clip_008,80.6,86.1,claro entonces yo ayer dije pucha porque siemp...
8,clip_009,79.8,68.0,porque porque está cerrado un proceso pero si ...
9,clip_010,67.8,57.5,"No se puede, no se puede, no se puede visualiz..."
